# 🚀 Project Atlas

# ⚡ Day 23 – PySpark DataFrame Transformations

**Date:** November 3, 2026

**Phase:** Phase 4 – PySpark

**Theme:** Transforming and Cleaning PySpark DataFrames

**Difficulty:** ⭐⭐⭐☆☆

---



# 🎯 Objective

In the previous session, I learned about Spark architecture, SparkSession, DataFrames, transformations, actions and lazy evaluation.

Today, I will take the next step and learn how to manipulate existing DataFrames.

In real-world data engineering, raw datasets are rarely ready for analysis. They often contain unnecessary columns, inconsistent values, missing information and data that needs to be transformed according to business requirements.

By the end of today's session, I should be able to add new columns, rename existing columns, remove unnecessary columns and apply basic conditional transformations using PySpark.

Today's session will also introduce basic missing-value handling.

---



# 📖 Part 1 – Understanding DataFrame Transformations

**Objective:** Understand how PySpark modifies and transforms existing DataFrames.

A DataFrame transformation is an operation that produces a new DataFrame from an existing DataFrame.

Examples include:

- Adding a calculated column.
- Renaming an existing column.
- Removing unnecessary columns.
- Applying conditions to column values.
- Replacing missing values.

An important concept to remember is that PySpark DataFrames are immutable.

This means that transformations do not directly modify the original DataFrame. Instead, they return a new DataFrame.

For example:

    df_new = df.withColumn("bonus", F.col("salary") * 0.10)

Here, the original `df` remains unchanged, while `df_new` contains the additional bonus column.

---



# 📖 Part 2 – Preparing the Employee Dataset

**Objective:** Create a practical dataset that we can use throughout today's session.


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ProjectAtlas_Day22")
    .master("local[*]")
    .getOrCreate()
)

print(spark.version)


4.2.0


In [4]:


from pyspark.sql import functions as F
employees = [
    (101, "Arun", "IT", 45000, 2),
    (102, "Priya", "HR", 35000, 1),
    (103, "Rahul", "IT", 65000, 5),
    (104, "Sneha", "Finance", 55000, 4),
    (105, "Kiran", "HR", None, 3),
    (106, "Divya", "Finance", 75000, 7)
]
columns = [
    "employee_id",
    "name",
    "department",
    "salary",
    "experience"
]
df = spark.createDataFrame(employees, columns)
df.show()
df.printSchema()


+-----------+-----+----------+------+----------+
|employee_id| name|department|salary|experience|
+-----------+-----+----------+------+----------+
|        101| Arun|        IT| 45000|         2|
|        102|Priya|        HR| 35000|         1|
|        103|Rahul|        IT| 65000|         5|
|        104|Sneha|   Finance| 55000|         4|
|        105|Kiran|        HR|  NULL|         3|
|        106|Divya|   Finance| 75000|         7|
+-----------+-----+----------+------+----------+

root
 |-- employee_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- experience: long (nullable = true)




**Observation:**

The dataset contains six employee records.

Notice that Kiran's salary is NULL. We will use this record later to understand missing-value handling.

---



# 📖 Part 3 – Adding and Modifying Columns

**Objective:** Learn how to create new columns and modify existing columns using `withColumn()`.

The `withColumn()` method is used to add a new column or replace an existing column with a transformed expression.

**Example 1 – Calculate Employee Bonus**

Business requirement:

The organization provides a bonus equal to 10% of an employee's salary.


In [5]:

df_bonus = df.withColumn(
        "bonus",
        F.col("salary") * 0.10
    )

df_bonus.show()


+-----------+-----+----------+------+----------+------+
|employee_id| name|department|salary|experience| bonus|
+-----------+-----+----------+------+----------+------+
|        101| Arun|        IT| 45000|         2|4500.0|
|        102|Priya|        HR| 35000|         1|3500.0|
|        103|Rahul|        IT| 65000|         5|6500.0|
|        104|Sneha|   Finance| 55000|         4|5500.0|
|        105|Kiran|        HR|  NULL|         3|  NULL|
|        106|Divya|   Finance| 75000|         7|7500.0|
+-----------+-----+----------+------+----------+------+




**Example 2 – Calculate Annual Salary**


In [6]:
df_annual = df.withColumn(
        "annual_salary",
        F.col("salary") * 12
    )

df_annual.show()


+-----------+-----+----------+------+----------+-------------+
|employee_id| name|department|salary|experience|annual_salary|
+-----------+-----+----------+------+----------+-------------+
|        101| Arun|        IT| 45000|         2|       540000|
|        102|Priya|        HR| 35000|         1|       420000|
|        103|Rahul|        IT| 65000|         5|       780000|
|        104|Sneha|   Finance| 55000|         4|       660000|
|        105|Kiran|        HR|  NULL|         3|         NULL|
|        106|Divya|   Finance| 75000|         7|       900000|
+-----------+-----+----------+------+----------+-------------+




**Example 3 – Modify an Existing Column**

Business requirement:

Apply a 5% salary increment.


In [7]:

df_incremented = df.withColumn(
        "salary",
        F.col("salary") * 1.05
    )

df_incremented.show()


+-----------+-----+----------+-------+----------+
|employee_id| name|department| salary|experience|
+-----------+-----+----------+-------+----------+
|        101| Arun|        IT|47250.0|         2|
|        102|Priya|        HR|36750.0|         1|
|        103|Rahul|        IT|68250.0|         5|
|        104|Sneha|   Finance|57750.0|         4|
|        105|Kiran|        HR|   NULL|         3|
|        106|Divya|   Finance|78750.0|         7|
+-----------+-----+----------+-------+----------+




**Observation:**

- `withColumn()` can create new columns.
- It can also replace existing columns.
- The original DataFrame remains unchanged.
- NULL values generally remain NULL during arithmetic calculations.

---



# 📖 Part 4 – Renaming and Removing Columns

**Objective:** Learn how to standardize column names and remove unnecessary fields.

**4.1 – Renaming Columns**

The `withColumnRenamed()` method changes the name of an existing column.



In [8]:
df_renamed = df.withColumnRenamed(
    "employee_id",
    "emp_id"
)
df_renamed.show()


+------+-----+----------+------+----------+
|emp_id| name|department|salary|experience|
+------+-----+----------+------+----------+
|   101| Arun|        IT| 45000|         2|
|   102|Priya|        HR| 35000|         1|
|   103|Rahul|        IT| 65000|         5|
|   104|Sneha|   Finance| 55000|         4|
|   105|Kiran|        HR|  NULL|         3|
|   106|Divya|   Finance| 75000|         7|
+------+-----+----------+------+----------+



We can also rename multiple columns:


In [9]:

df_renamed = (
        df
        .withColumnRenamed("employee_id", "emp_id")
        .withColumnRenamed("experience", "years_of_experience")
    )

df_renamed.printSchema()


root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- years_of_experience: long (nullable = true)




**4.2 – Removing Columns**

The `drop()` method removes unnecessary columns.


In [10]:

df_dropped = df.drop("experience")
df_dropped.show()


+-----------+-----+----------+------+
|employee_id| name|department|salary|
+-----------+-----+----------+------+
|        101| Arun|        IT| 45000|
|        102|Priya|        HR| 35000|
|        103|Rahul|        IT| 65000|
|        104|Sneha|   Finance| 55000|
|        105|Kiran|        HR|  NULL|
|        106|Divya|   Finance| 75000|
+-----------+-----+----------+------+




To remove multiple columns:


In [11]:

df_dropped = df.drop(
        "experience",
        "department"
    )

df_dropped.show()


+-----------+-----+------+
|employee_id| name|salary|
+-----------+-----+------+
|        101| Arun| 45000|
|        102|Priya| 35000|
|        103|Rahul| 65000|
|        104|Sneha| 55000|
|        105|Kiran|  NULL|
|        106|Divya| 75000|
+-----------+-----+------+




**Observation:**

These operations return new DataFrames with the modified schemas.

---



# 📖 Part 5 – Conditional Transformations

**Objective:** Learn how to apply business rules using `when()` and `otherwise()`.

In real-world datasets, we frequently need to categorize records based on specific conditions.

PySpark provides `when()` and `otherwise()` to implement this logic.

**Example 1 – Salary Classification**

Business requirement:

- Salary greater than or equal to 60000 → High.
- Salary below 60000 → Standard.


In [12]:

df_category = df.withColumn(
        "salary_category",
        F.when(F.col("salary") >= 60000, "High")
         .otherwise("Standard")
    )

df_category.show()


+-----------+-----+----------+------+----------+---------------+
|employee_id| name|department|salary|experience|salary_category|
+-----------+-----+----------+------+----------+---------------+
|        101| Arun|        IT| 45000|         2|       Standard|
|        102|Priya|        HR| 35000|         1|       Standard|
|        103|Rahul|        IT| 65000|         5|           High|
|        104|Sneha|   Finance| 55000|         4|       Standard|
|        105|Kiran|        HR|  NULL|         3|       Standard|
|        106|Divya|   Finance| 75000|         7|           High|
+-----------+-----+----------+------+----------+---------------+




**Example 2 – Experience Classification**

Business requirement:

- Experience less than or equal to 2 years → Junior.
- Experience between 3 and 5 years → Mid-Level.
- Experience above 5 years → Senior.


In [13]:

df_experience = df.withColumn(
        "experience_level",
        F.when(F.col("experience") <= 2, "Junior")
         .when(F.col("experience") <= 5, "Mid-Level")
         .otherwise("Senior")
    )

df_experience.show()


+-----------+-----+----------+------+----------+----------------+
|employee_id| name|department|salary|experience|experience_level|
+-----------+-----+----------+------+----------+----------------+
|        101| Arun|        IT| 45000|         2|          Junior|
|        102|Priya|        HR| 35000|         1|          Junior|
|        103|Rahul|        IT| 65000|         5|       Mid-Level|
|        104|Sneha|   Finance| 55000|         4|       Mid-Level|
|        105|Kiran|        HR|  NULL|         3|       Mid-Level|
|        106|Divya|   Finance| 75000|         7|          Senior|
+-----------+-----+----------+------+----------+----------------+




**Example 3 – Explicitly Identify Missing Salary**


In [14]:

df_status = df.withColumn(
        "salary_status",
        F.when(F.col("salary").isNull(), "Missing")
         .otherwise("Available")
    )

df_status.show()


+-----------+-----+----------+------+----------+-------------+
|employee_id| name|department|salary|experience|salary_status|
+-----------+-----+----------+------+----------+-------------+
|        101| Arun|        IT| 45000|         2|    Available|
|        102|Priya|        HR| 35000|         1|    Available|
|        103|Rahul|        IT| 65000|         5|    Available|
|        104|Sneha|   Finance| 55000|         4|    Available|
|        105|Kiran|        HR|  NULL|         3|      Missing|
|        106|Divya|   Finance| 75000|         7|    Available|
+-----------+-----+----------+------+----------+-------------+



**Observation:**

Conditional transformations allow us to implement business rules directly within DataFrame operations.

---



# 📖 Part 6 – Introduction to Missing-Value Handling

**Objective:** Understand how to identify and handle NULL values in PySpark.

Real-world datasets frequently contain missing values.

PySpark provides the `na` interface to handle these values.

**6.1 – Identifying NULL Values**


In [15]:

df.filter(F.col("salary").isNull()).show()


+-----------+-----+----------+------+----------+
|employee_id| name|department|salary|experience|
+-----------+-----+----------+------+----------+
|        105|Kiran|        HR|  NULL|         3|
+-----------+-----+----------+------+----------+



To identify records with available salaries:


In [16]:

df.filter(F.col("salary").isNotNull()).show()


+-----------+-----+----------+------+----------+
|employee_id| name|department|salary|experience|
+-----------+-----+----------+------+----------+
|        101| Arun|        IT| 45000|         2|
|        102|Priya|        HR| 35000|         1|
|        103|Rahul|        IT| 65000|         5|
|        104|Sneha|   Finance| 55000|         4|
|        106|Divya|   Finance| 75000|         7|
+-----------+-----+----------+------+----------+




**6.2 – Filling Missing Values**

The `na.fill()` method replaces NULL values with a specified value.


In [17]:

df_filled = df.na.fill(
        0,
        subset=["salary"]
    )

df_filled.show()


+-----------+-----+----------+------+----------+
|employee_id| name|department|salary|experience|
+-----------+-----+----------+------+----------+
|        101| Arun|        IT| 45000|         2|
|        102|Priya|        HR| 35000|         1|
|        103|Rahul|        IT| 65000|         5|
|        104|Sneha|   Finance| 55000|         4|
|        105|Kiran|        HR|     0|         3|
|        106|Divya|   Finance| 75000|         7|
+-----------+-----+----------+------+----------+




**6.3 – Removing Records with Missing Values**

The `na.drop()` method removes rows containing NULL values.


In [18]:

df_clean = df.na.drop()
df_clean.show()


+-----------+-----+----------+------+----------+
|employee_id| name|department|salary|experience|
+-----------+-----+----------+------+----------+
|        101| Arun|        IT| 45000|         2|
|        102|Priya|        HR| 35000|         1|
|        103|Rahul|        IT| 65000|         5|
|        104|Sneha|   Finance| 55000|         4|
|        106|Divya|   Finance| 75000|         7|
+-----------+-----+----------+------+----------+




We can also restrict the operation to a specific column:


In [19]:

df_clean = df.na.drop(
        subset=["salary"]
    )

df_clean.show()


+-----------+-----+----------+------+----------+
|employee_id| name|department|salary|experience|
+-----------+-----+----------+------+----------+
|        101| Arun|        IT| 45000|         2|
|        102|Priya|        HR| 35000|         1|
|        103|Rahul|        IT| 65000|         5|
|        104|Sneha|   Finance| 55000|         4|
|        106|Divya|   Finance| 75000|         7|
+-----------+-----+----------+------+----------+




**6.4 – Replacing Specific Values**

The `na.replace()` method replaces specified existing values.

For example, standardizing the department value `Tech` to `IT`:



In [20]:
df_replaced = df.na.replace(
        "Tech",
        "IT",
        subset=["department"]
    )

df_replaced.show()


+-----------+-----+----------+------+----------+
|employee_id| name|department|salary|experience|
+-----------+-----+----------+------+----------+
|        101| Arun|        IT| 45000|         2|
|        102|Priya|        HR| 35000|         1|
|        103|Rahul|        IT| 65000|         5|
|        104|Sneha|   Finance| 55000|         4|
|        105|Kiran|        HR|  NULL|         3|
|        106|Divya|   Finance| 75000|         7|
+-----------+-----+----------+------+----------+




**Observation:**

- `na.fill()` replaces NULL values.
- `na.drop()` removes rows containing NULL values.
- `na.replace()` replaces specified values.

The appropriate approach depends on the business requirement. Missing values should not automatically be replaced with zero without understanding their meaning.

---



# 🧩 Mini Check

Without copying the previous transformation pipeline:

Create a new employee DataFrame using the following records:


In [22]:

employees = [
        (101, "Arun", "IT", 45000, 2),
        (102, "Priya", "HR", 35000, 1),
        (103, "Rahul", "Tech", 65000, 5),
        (104, "Sneha", "Finance", 55000, 4),
        (105, "Kiran", "HR", None, 3),
        (106, "Divya", "Finance", 75000, 7),
        (107, "Vikram", "Tech", 80000, 8)
    ]

columns = [
        "employee_id",
        "name",
        "department",
        "salary",
        "experience"
    ]

df_challenge = spark.createDataFrame(
        employees,
        columns
    )



**Your Tasks:**

1. Rename `employee_id` to `emp_id`.
2. Replace `Tech` with `IT` in the department column.
3. Fill missing salary values with zero.
4. Create an `annual_salary` column.
5. Create a `bonus` column equal to 10% of salary.
6. Categorize employees based on experience:
   - 0–2 years: Junior.
   - 3–5 years: Mid-Level.
   - Above 5 years: Senior.
7. Create a `salary_category` column:
   - Salary >= 60000: High.
   - Salary < 60000: Standard.
8. Remove the department column.
9. Display the final DataFrame.
10. Print the final schema.

**Additional Challenge:**

Before filling missing salaries, create a `salary_status` column to identify whether the original salary was Missing or Available.

---


In [23]:
df_challenge = df_challenge.withColumnRenamed("employee_id","emp_id")
df_challenge.show()

+------+------+----------+------+----------+
|emp_id|  name|department|salary|experience|
+------+------+----------+------+----------+
|   101|  Arun|        IT| 45000|         2|
|   102| Priya|        HR| 35000|         1|
|   103| Rahul|      Tech| 65000|         5|
|   104| Sneha|   Finance| 55000|         4|
|   105| Kiran|        HR|  NULL|         3|
|   106| Divya|   Finance| 75000|         7|
|   107|Vikram|      Tech| 80000|         8|
+------+------+----------+------+----------+



In [24]:
df_challenge = df_challenge.replace("Tech","IT",subset=["department"])
df_challenge.show()

+------+------+----------+------+----------+
|emp_id|  name|department|salary|experience|
+------+------+----------+------+----------+
|   101|  Arun|        IT| 45000|         2|
|   102| Priya|        HR| 35000|         1|
|   103| Rahul|        IT| 65000|         5|
|   104| Sneha|   Finance| 55000|         4|
|   105| Kiran|        HR|  NULL|         3|
|   106| Divya|   Finance| 75000|         7|
|   107|Vikram|        IT| 80000|         8|
+------+------+----------+------+----------+



In [25]:
df_challenge = df_challenge.na.fill(0,subset=["salary"])
df_challenge.show()

+------+------+----------+------+----------+
|emp_id|  name|department|salary|experience|
+------+------+----------+------+----------+
|   101|  Arun|        IT| 45000|         2|
|   102| Priya|        HR| 35000|         1|
|   103| Rahul|        IT| 65000|         5|
|   104| Sneha|   Finance| 55000|         4|
|   105| Kiran|        HR|     0|         3|
|   106| Divya|   Finance| 75000|         7|
|   107|Vikram|        IT| 80000|         8|
+------+------+----------+------+----------+



In [26]:
df_challenge = df_challenge.withColumn("Annual_salary",
                                       F.col("salary")*12)
df_challenge.show()

+------+------+----------+------+----------+-------------+
|emp_id|  name|department|salary|experience|Annual_salary|
+------+------+----------+------+----------+-------------+
|   101|  Arun|        IT| 45000|         2|       540000|
|   102| Priya|        HR| 35000|         1|       420000|
|   103| Rahul|        IT| 65000|         5|       780000|
|   104| Sneha|   Finance| 55000|         4|       660000|
|   105| Kiran|        HR|     0|         3|            0|
|   106| Divya|   Finance| 75000|         7|       900000|
|   107|Vikram|        IT| 80000|         8|       960000|
+------+------+----------+------+----------+-------------+



In [27]:
df_challenge = df_challenge.withColumn("Bonus",
                                       F.col("salary")*1.1)
df_challenge.show()

+------+------+----------+------+----------+-------------+-----------------+
|emp_id|  name|department|salary|experience|Annual_salary|            Bonus|
+------+------+----------+------+----------+-------------+-----------------+
|   101|  Arun|        IT| 45000|         2|       540000|49500.00000000001|
|   102| Priya|        HR| 35000|         1|       420000|          38500.0|
|   103| Rahul|        IT| 65000|         5|       780000|          71500.0|
|   104| Sneha|   Finance| 55000|         4|       660000|60500.00000000001|
|   105| Kiran|        HR|     0|         3|            0|              0.0|
|   106| Divya|   Finance| 75000|         7|       900000|          82500.0|
|   107|Vikram|        IT| 80000|         8|       960000|          88000.0|
+------+------+----------+------+----------+-------------+-----------------+



In [29]:
df_challenge = df_challenge.withColumn("Category",
                                       F.when(F.col("experience")<=2,"Junior")
                                       .when(F.col("experience")<=5,"Mid-Level").otherwise("Senior"))
df_challenge.show()

+------+------+----------+------+----------+-------------+-----------------+---------+
|emp_id|  name|department|salary|experience|Annual_salary|            Bonus| Category|
+------+------+----------+------+----------+-------------+-----------------+---------+
|   101|  Arun|        IT| 45000|         2|       540000|49500.00000000001|   Junior|
|   102| Priya|        HR| 35000|         1|       420000|          38500.0|   Junior|
|   103| Rahul|        IT| 65000|         5|       780000|          71500.0|Mid-Level|
|   104| Sneha|   Finance| 55000|         4|       660000|60500.00000000001|Mid-Level|
|   105| Kiran|        HR|     0|         3|            0|              0.0|Mid-Level|
|   106| Divya|   Finance| 75000|         7|       900000|          82500.0|   Senior|
|   107|Vikram|        IT| 80000|         8|       960000|          88000.0|   Senior|
+------+------+----------+------+----------+-------------+-----------------+---------+



In [30]:
df_challenge = df_challenge.withColumn("Salary_Category",
                                       F.when(F.col("salary")>=60000,"High")
                                       .otherwise("Standard"))
df_challenge.show()

+------+------+----------+------+----------+-------------+-----------------+---------+---------------+
|emp_id|  name|department|salary|experience|Annual_salary|            Bonus| Category|Salary_Category|
+------+------+----------+------+----------+-------------+-----------------+---------+---------------+
|   101|  Arun|        IT| 45000|         2|       540000|49500.00000000001|   Junior|       Standard|
|   102| Priya|        HR| 35000|         1|       420000|          38500.0|   Junior|       Standard|
|   103| Rahul|        IT| 65000|         5|       780000|          71500.0|Mid-Level|           High|
|   104| Sneha|   Finance| 55000|         4|       660000|60500.00000000001|Mid-Level|       Standard|
|   105| Kiran|        HR|     0|         3|            0|              0.0|Mid-Level|       Standard|
|   106| Divya|   Finance| 75000|         7|       900000|          82500.0|   Senior|           High|
|   107|Vikram|        IT| 80000|         8|       960000|          88000

In [31]:
df_challenge = df_challenge.drop("department")
df_challenge.show()

+------+------+------+----------+-------------+-----------------+---------+---------------+
|emp_id|  name|salary|experience|Annual_salary|            Bonus| Category|Salary_Category|
+------+------+------+----------+-------------+-----------------+---------+---------------+
|   101|  Arun| 45000|         2|       540000|49500.00000000001|   Junior|       Standard|
|   102| Priya| 35000|         1|       420000|          38500.0|   Junior|       Standard|
|   103| Rahul| 65000|         5|       780000|          71500.0|Mid-Level|           High|
|   104| Sneha| 55000|         4|       660000|60500.00000000001|Mid-Level|       Standard|
|   105| Kiran|     0|         3|            0|              0.0|Mid-Level|       Standard|
|   106| Divya| 75000|         7|       900000|          82500.0|   Senior|           High|
|   107|Vikram| 80000|         8|       960000|          88000.0|   Senior|           High|
+------+------+------+----------+-------------+-----------------+---------+-----

In [32]:
df_challenge.printSchema()

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- experience: long (nullable = true)
 |-- Annual_salary: long (nullable = true)
 |-- Bonus: double (nullable = true)
 |-- Category: string (nullable = false)
 |-- Salary_Category: string (nullable = false)




# 📚 Day Summary

Today I progressed from basic PySpark DataFrame operations to practical DataFrame transformations.

I learned how to create and modify columns using `withColumn()`, rename columns using `withColumnRenamed()`, and remove unnecessary columns using `drop()`.

I also explored conditional transformations using `when()` and `otherwise()`, along with the basics of identifying, filling, removing and replacing missing values.

The main objective was to understand how raw employee data can be transformed according to business requirements.

I will reinforce these concepts by completing the Employee Data Transformation Mini Check independently.